# Torn-meniscus patients: model predictions vs hand segmentation

Per patient, **every slice where any of the three masks has a label**, side by side:
Original | Replica | Crop+Seq | Overlay (all three) | Your segmentation.

* Everything is binary: *meniscus* vs *no meniscus* (model lateral/medial labels are merged).
* **Outlines only, no fill.** Red = Replica, cyan = Crop+Seq, yellow = your segmentation.
* Section A per patient = full view; section B = zoomed to the union of all masks for that patient (square crop, same crop on every slice).
* Predictions come from `results/torn_predictions_{replica_seed42,crop_sequence}/` (no models are run here).
* Your masks are pushed through the same RAS-reorient / isotropic-resample / resize-to-384 steps as the images (nearest-neighbour).
* Patients whose raw PD scan can't be found under `RAW_DIRS` are skipped with a message: add the folder to `RAW_DIRS`. (All 5 raw scans are in `~/Desktop/delete/aug 22`.)

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
import matplotlib.pyplot as plt
from torn_compare_utils import load_patient, plot_patient, summary_table, COLORS, dice
import pandas as pd
pd.set_option("display.width", 200)

REPLICA_DIR = ROOT / "results/torn_predictions_replica_seed42"
CROPSEQ_DIR = ROOT / "results/torn_predictions_crop_sequence"
SEG_ROOT    = Path("/Users/anshikabajpai/Desktop/delete/nikhil-segmentations 2")   # <id>/Segmentation.seg.nrrd
RAW_DIRS    = [SEG_ROOT, Path("/Users/anshikabajpai/Desktop/delete/aug 22")]   # raw PD scans, searched recursively (aug 21 is an identical copy)

PATIENTS = ['AC13D01F99D88C', 'AC116435D25299', 'AC1CEC4C14F850', 'AC152F0CF07CF3', 'AC1A200F61EDC1']
DATA = {p: load_patient(p, REPLICA_DIR, CROPSEQ_DIR, SEG_ROOT, RAW_DIRS) for p in PATIENTS}
DATA = {p: d for p, d in DATA.items() if d is not None}
print("loaded:", list(DATA))

## Overall Dice vs your segmentation (binary, 3D)

In [ ]:
rows = [{"patient": p, "manual_slices": int(d["manual"].any((1,2)).sum()),
         "dice_replica": dice(d["replica"], d["manual"]), "dice_crop_seq": dice(d["crop_seq"], d["manual"])}
        for p, d in DATA.items()]
pd.DataFrame(rows).set_index("patient").round(3)

---
## AC13D01F99D88C

In [ ]:
d = DATA.get("AC13D01F99D88C")
if d is None: print("not loaded (see message above)")
else: display(summary_table(d))

### AC13D01F99D88C — A. full view (outlines)

In [ ]:
d = DATA.get("AC13D01F99D88C")
if d is not None:
    fig = plot_patient("AC13D01F99D88C", d, zoom_in=False); plt.show()

### AC13D01F99D88C — B. zoomed (outlines)

In [ ]:
d = DATA.get("AC13D01F99D88C")
if d is not None:
    fig = plot_patient("AC13D01F99D88C", d, zoom_in=True); plt.show()

---
## AC116435D25299

In [ ]:
d = DATA.get("AC116435D25299")
if d is None: print("not loaded (see message above)")
else: display(summary_table(d))

### AC116435D25299 — A. full view (outlines)

In [ ]:
d = DATA.get("AC116435D25299")
if d is not None:
    fig = plot_patient("AC116435D25299", d, zoom_in=False); plt.show()

### AC116435D25299 — B. zoomed (outlines)

In [ ]:
d = DATA.get("AC116435D25299")
if d is not None:
    fig = plot_patient("AC116435D25299", d, zoom_in=True); plt.show()

---
## AC1CEC4C14F850

In [ ]:
d = DATA.get("AC1CEC4C14F850")
if d is None: print("not loaded (see message above)")
else: display(summary_table(d))

### AC1CEC4C14F850 — A. full view (outlines)

In [ ]:
d = DATA.get("AC1CEC4C14F850")
if d is not None:
    fig = plot_patient("AC1CEC4C14F850", d, zoom_in=False); plt.show()

### AC1CEC4C14F850 — B. zoomed (outlines)

In [ ]:
d = DATA.get("AC1CEC4C14F850")
if d is not None:
    fig = plot_patient("AC1CEC4C14F850", d, zoom_in=True); plt.show()

---
## AC152F0CF07CF3

In [ ]:
d = DATA.get("AC152F0CF07CF3")
if d is None: print("not loaded (see message above)")
else: display(summary_table(d))

### AC152F0CF07CF3 — A. full view (outlines)

In [ ]:
d = DATA.get("AC152F0CF07CF3")
if d is not None:
    fig = plot_patient("AC152F0CF07CF3", d, zoom_in=False); plt.show()

### AC152F0CF07CF3 — B. zoomed (outlines)

In [ ]:
d = DATA.get("AC152F0CF07CF3")
if d is not None:
    fig = plot_patient("AC152F0CF07CF3", d, zoom_in=True); plt.show()

---
## AC1A200F61EDC1

In [ ]:
d = DATA.get("AC1A200F61EDC1")
if d is None: print("not loaded (see message above)")
else: display(summary_table(d))

### AC1A200F61EDC1 — A. full view (outlines)

In [ ]:
d = DATA.get("AC1A200F61EDC1")
if d is not None:
    fig = plot_patient("AC1A200F61EDC1", d, zoom_in=False); plt.show()

### AC1A200F61EDC1 — B. zoomed (outlines)

In [ ]:
d = DATA.get("AC1A200F61EDC1")
if d is not None:
    fig = plot_patient("AC1A200F61EDC1", d, zoom_in=True); plt.show()

---
# Novel approaches on the torn patients (vs your segmentation)

**Shown only for `AC1CEC4C14F850`** (`NOVEL_PATIENTS` in the first cell of this section; add IDs there to widen).

Same 5 patients, now with the Phase 5 approaches. **GT = your segmentation** (binary). Per patient: one row per slice where your seg has meniscus;
columns = Image | baseline | approaches | GT. **No overlays** — each mask column is the image with only that mask's boundary outline (no fill); GT is yellow.
Zoom = one square crop per patient around GT + all shown predictions. Column title = patient 3D Dice vs your seg; label under each panel = that slice's Dice.

* **Crop+seq arm:** crop+seq baseline vs A3-up, A3-down, E1, E2, G2, D (`results/torn_predictions_crop_*`).
* **Plain 2.5D (replica) arm:** replica baseline vs A3v2-up, A3v2-down, E1, E2, G2, D (`results/torn_predictions_2_5d_*`).
* A run whose prediction folder is missing is dropped with a message (pull it with `scp` first).

In [ ]:
from phase5_slice_compare import plot_patient_comparison
from torn_compare_utils import find_one, load_pred

RES = ROOT / "results"
# Novel-approach comparisons below are shown ONLY for these patients (baselines replica + crop+seq are in every figure).
NOVEL_PATIENTS = ["AC1CEC4C14F850"]   # add more IDs to widen
CROP_RUNS  = {"crop+seq (base)": None, "A3-up": "crop_a3_up", "A3-down": "crop_a3_down", "E1": "crop_e1_pos",
              "E2": "crop_e2_film", "G2": "crop_g2_rim", "D": "crop_d_shape"}
PLAIN_RUNS = {"replica (base)": None, "A3v2-up": "2_5d_a3v2_up", "A3v2-down": "2_5d_a3v2_down", "E1": "2_5d_e1",
              "E2": "2_5d_e2", "G2": "2_5d_g2", "D": "2_5d_d"}

def load_arm(runs, baseline_key):
    """{pid: {run name: bool (n,384,384)}}; the baseline comes from the already-loaded DATA."""
    out = {}
    for pid, d in DATA.items():
        if pid not in NOVEL_PATIENTS: continue
        preds = {}
        for name, tag in runs.items():
            if tag is None:
                preds[name] = d[baseline_key]
                continue
            folder = RES / f"torn_predictions_{tag}"
            f = find_one(folder, f"{pid}*_meniscus_pred.nii.gz") if folder.exists() else None
            if f is None:
                print(f"[{pid}] {name}: no prediction in {folder.name} -> column dropped")
                continue
            preds[name] = load_pred(f)
        out[pid] = preds
    return out

def arm_dice_table(arm):
    rows = {pid: {k: dice(v, DATA[pid]["manual"]) for k, v in preds.items()} for pid, preds in arm.items()}
    t = pd.DataFrame(rows).T
    if len(t) > 1:
        t.loc["MEAN"] = t.mean()
    return t.round(3)

def show_arm(arm, title, **plot_kw):
    print(title)
    for pid, preds in arm.items():
        d = DATA[pid]
        pdice = {k: dice(v, d["manual"]) for k, v in preds.items()}
        print(f"\n### {pid}")
        plot_patient_comparison(pid, d["vol"], d["manual"], preds, pdice, **plot_kw)
        plt.show()

## Crop+seq arm

In [ ]:
crop_arm = load_arm(CROP_RUNS, "crop_seq")
display(arm_dice_table(crop_arm))

In [ ]:
show_arm(crop_arm, "crop+seq arm (GT = your segmentation)")

## Plain 2.5D (replica) arm

In [ ]:
plain_arm = load_arm(PLAIN_RUNS, "replica")
display(arm_dice_table(plain_arm))

In [ ]:
show_arm(plain_arm, "plain 2.5D arm (GT = your segmentation)")

---
# G2 and E (E1, E2) in both arms, with both baselines

One cell, all 5 torn patients. To keep it readable each patient gets **two figures** (6 columns each) instead of one with 10:

* **Plain 2.5D arm:** Image | replica | G2 | E1 | E2 | GT (your seg)
* **Crop+seq arm:** Image | crop+seq | G2 | E1 | E2 | GT (your seg)

Together that is 2 G2 runs and 4 E runs next to the two baselines. Tighter zoom than before (`ZOOM_PAD`) and larger panels (`PANEL`); one row per slice where your seg has meniscus; boundary outlines only (no fill, no overlay), GT yellow.
Column title = patient 3D Dice vs your seg; label under each panel = that slice's Dice. Lower `ZOOM_PAD` to zoom in more, raise `PANEL` to enlarge.

In [ ]:
# name -> ("base", key in DATA)  or  results/torn_predictions_<tag>
GE_RUNS = {
    "replica":        ("base", "replica"),
    "G2 (plain)":     "2_5d_g2",
    "E1 (plain)":     "2_5d_e1",
    "E2 (plain)":     "2_5d_e2",
    "crop+seq":       ("base", "crop_seq"),
    "G2 (crop)":      "crop_g2_rim",
    "E1 (crop)":      "crop_e1_pos",
    "E2 (crop)":      "crop_e2_film",
}

ge_arm = {}
for pid, d in DATA.items():
    if pid not in NOVEL_PATIENTS: continue
    preds = {}
    for name, src in GE_RUNS.items():
        if isinstance(src, tuple):
            preds[name] = d[src[1]]
            continue
        folder = RES / f"torn_predictions_{src}"
        f = find_one(folder, f"{pid}*_meniscus_pred.nii.gz") if folder.exists() else None
        if f is None:
            print(f"[{pid}] {name}: no prediction in {folder.name} -> column dropped"); continue
        preds[name] = load_pred(f)
    ge_arm[pid] = preds

display(arm_dice_table(ge_arm))

ZOOM_PAD, PANEL = 10, 3.6
PLAIN_COLS = ["replica", "G2 (plain)", "E1 (plain)", "E2 (plain)"]
CROP_COLS  = ["crop+seq", "G2 (crop)", "E1 (crop)", "E2 (crop)"]
for cols, label in ((PLAIN_COLS, "PLAIN 2.5D arm"), (CROP_COLS, "CROP+SEQ arm")):
    sub = {pid: {k: v for k, v in preds.items() if k in cols} for pid, preds in ge_arm.items()}
    show_arm(sub, f"{label}: baseline + G2 + E1 + E2 vs GT (your segmentation)", pad=ZOOM_PAD, panel=PANEL)

---
# A3 (up / down) in both arms, with both baselines

Same layout as the G2/E cell: two figures per patient, zoomed, one row per slice where your seg has meniscus, outlines only (no fill, no overlay), GT yellow.

* **Plain 2.5D arm:** Image | replica | A3v2-up | A3v2-down | GT (A3v2 = the CE-fixed version)
* **Crop+seq arm:** Image | crop+seq | A3-up | A3-down | GT (these carry the CE-normalisation bug, so they are not clean position-weighting results)

Only up/down were run on the torn patients. The A3v2-none control and the A3 v1 runs were not in the torn inference job.
Column title = patient 3D Dice vs your seg; label under each panel = that slice's Dice.

In [ ]:
def load_runs(runs):
    """runs: {name: ("base", key in DATA) or results/torn_predictions_<tag>} -> {pid: {name: bool vol}}"""
    out = {}
    for pid, d in DATA.items():
        if pid not in NOVEL_PATIENTS: continue
        preds = {}
        for name, src in runs.items():
            if isinstance(src, tuple):
                preds[name] = d[src[1]]; continue
            folder = RES / f"torn_predictions_{src}"
            f = find_one(folder, f"{pid}*_meniscus_pred.nii.gz") if folder.exists() else None
            if f is None:
                print(f"[{pid}] {name}: no prediction in {folder.name} -> column dropped"); continue
            preds[name] = load_pred(f)
        out[pid] = preds
    return out

A3_RUNS = {
    "replica": ("base", "replica"), "A3v2-up": "2_5d_a3v2_up", "A3v2-down": "2_5d_a3v2_down",
    "crop+seq": ("base", "crop_seq"), "A3-up (crop)": "crop_a3_up", "A3-down (crop)": "crop_a3_down",
}
a3_arm = load_runs(A3_RUNS)
display(arm_dice_table(a3_arm))

ZOOM_PAD, PANEL = 10, 3.6
A3_PLAIN = ["replica", "A3v2-up", "A3v2-down"]
A3_CROP  = ["crop+seq", "A3-up (crop)", "A3-down (crop)"]
for cols, label in ((A3_PLAIN, "PLAIN 2.5D arm"), (A3_CROP, "CROP+SEQ arm")):
    sub = {pid: {k: v for k, v in preds.items() if k in cols} for pid, preds in a3_arm.items()}
    show_arm(sub, f"{label}: baseline + A3 up/down vs GT (your segmentation)", pad=ZOOM_PAD, panel=PANEL)

---
# D (shape prior) in both arms, with both baselines

One figure per patient (only 6 columns): Image | replica | D (plain) | crop+seq | D (crop) | GT (your seg).
Zoomed, one row per slice where your seg has meniscus, outlines only, GT yellow. Note D's shape term fades to almost nothing within a few epochs, so D should be close to its baseline.

In [ ]:
def load_runs(runs):
    """runs: {name: ("base", key in DATA) or results/torn_predictions_<tag>} -> {pid: {name: bool vol}}"""
    out = {}
    for pid, d in DATA.items():
        if pid not in NOVEL_PATIENTS: continue
        preds = {}
        for name, src in runs.items():
            if isinstance(src, tuple):
                preds[name] = d[src[1]]; continue
            folder = RES / f"torn_predictions_{src}"
            f = find_one(folder, f"{pid}*_meniscus_pred.nii.gz") if folder.exists() else None
            if f is None:
                print(f"[{pid}] {name}: no prediction in {folder.name} -> column dropped"); continue
            preds[name] = load_pred(f)
        out[pid] = preds
    return out

D_RUNS = {"replica": ("base", "replica"), "D (plain)": "2_5d_d", "crop+seq": ("base", "crop_seq"), "D (crop)": "crop_d_shape"}
d_arm = load_runs(D_RUNS)
display(arm_dice_table(d_arm))
ZOOM_PAD, PANEL = 10, 3.6
show_arm(d_arm, "D (shape prior), both arms + both baselines vs GT (your segmentation)", pad=ZOOM_PAD, panel=PANEL)

---
# Phase 6 (v14 tear-amodal) vs replica, one held-out torn patient

Patient = `NOVEL_PATIENTS` (set in the cell that starts `from phase5_slice_compare import ...`; currently `AC1CEC4C14F850`).
These torn patients were **held out** of v14 training (the training script aborts if any of them is in train/val).

Columns: Image | replica | v14 R1 (gpu-interactive) | R1 | R2 | R3 | R4 | R1c | GT (your segmentation). A run whose
`results/torn_predictions_v14_<tag>/` folder has not been pulled yet is dropped with a message.

| Run | What it adds on top of the replica recipe |
|---|---|
| R1 / R1_int | synthetic tears painted inside the meniscus on training slices, labels unchanged (whole wedge) |
| R2 | R1 + visible-meniscus head |
| R3 | R2 + contour head |
| R4 | R3 + inclusion loss (visible inside whole) |
| R1c | R1 + contour head only |

Zoomed, one row per slice where your seg has meniscus, outlines only, GT yellow. Column title = 3D Dice vs your seg.

In [ ]:
V14_RUNS = {
    "replica":  ("base", "replica"),
    "R1 (int)": "v14_R1_int",
    "R1":       "v14_R1",
    "R2":       "v14_R2",
    "R3":       "v14_R3",
    "R4":       "v14_R4",
    "R1c":      "v14_R1c",
}

def load_v14(runs, patients):
    out = {}
    for pid in patients:
        d = DATA[pid]
        preds = {}
        for name, src in runs.items():
            if isinstance(src, tuple):
                preds[name] = d[src[1]]; continue
            folder = RES / f"torn_predictions_{src}"
            f = find_one(folder, f"{pid}*_meniscus_pred.nii.gz") if folder.exists() else None
            if f is None:
                print(f"[{pid}] {name}: not pulled yet ({folder.name}) -> column dropped"); continue
            preds[name] = load_pred(f)
        out[pid] = preds
    return out

v14_arm = load_v14(V14_RUNS, [p for p in NOVEL_PATIENTS if p in DATA])
display(arm_dice_table(v14_arm))
show_arm(v14_arm, "v14 tear-amodal runs vs replica (GT = your segmentation)", pad=10, panel=3.6)

### Tear maps (runs with a visible head: R2, R3, R4)

Per slice: Image | whole wedge (red) | visible meniscus (cyan) | tear map = p(whole) − p(visible), shown as a heat overlay,
with your segmentation (yellow) for reference. The tear map is the model's own guess at where the tear is; it was never trained on tear labels.

In [ ]:
import numpy as np
import nibabel as nib

def show_tearmaps(pid, tags=("v14_R2", "v14_R3", "v14_R4"), pad=10, panel=3.2):
    d = DATA[pid]
    for tag in tags:
        folder = RES / f"torn_predictions_{tag}"
        fw = find_one(folder, f"{pid}*_meniscus_pred.nii.gz") if folder.exists() else None
        fv = find_one(folder, f"{pid}*_visible.nii.gz") if folder.exists() else None
        ft = find_one(folder, f"{pid}*_tearmap.nii.gz") if folder.exists() else None
        if fw is None or fv is None or ft is None:
            print(f"[{pid}] {tag}: no visible/tearmap outputs yet -> skipped"); continue
        whole, vis = load_pred(fw), load_pred(fv)
        tear = np.asarray(nib.load(str(ft)).dataobj).astype(np.float32)
        rows = np.where(d["manual"].any((1, 2)))[0]
        m = whole | d["manual"]
        a, s = np.where(m.any(0))
        a0, a1 = max(a.min() - pad, 0), min(a.max() + pad + 1, 384)
        s0, s1 = max(s.min() - pad, 0), min(s.max() + pad + 1, 384)
        fig, axes = plt.subplots(len(rows), 4, figsize=(panel * 4, panel * len(rows)), squeeze=False)
        for r, sl in enumerate(rows):
            img = d["vol"][sl, a0:a1, s0:s1]
            for c in range(4):
                ax = axes[r, c]; ax.imshow(img, cmap="gray", vmin=0, vmax=1); ax.set_xticks([]); ax.set_yticks([])
            def outline(ax, msk, col):
                if msk.any(): ax.contour(msk.astype(float), levels=[0.5], colors=[col], linewidths=1.4)
            outline(axes[r, 1], whole[sl, a0:a1, s0:s1], "#ff2a2a")
            outline(axes[r, 2], vis[sl, a0:a1, s0:s1], "#00e5ff")
            axes[r, 3].imshow(np.ma.masked_less(tear[sl, a0:a1, s0:s1], 0.1), cmap="magma", vmin=0, vmax=1, alpha=0.8)
            outline(axes[r, 3], d["manual"][sl, a0:a1, s0:s1], "#ffe600")
            axes[r, 0].set_ylabel(f"slice {sl}", fontsize=10)
            if r == 0:
                for c, t in enumerate(["Image", "whole wedge", "visible", "tear map + your seg"]):
                    axes[r, c].set_title(t, fontsize=11)
        fig.suptitle(f"{pid}  {tag}  (tear map voxels > 0.5: {int((tear > 0.5).sum())})", fontsize=12, y=1.0)
        fig.tight_layout(); plt.show()

for pid in [p for p in NOVEL_PATIENTS if p in DATA]:
    show_tearmaps(pid)